# Lecture 6: Composite Systems and Entanglement
### Computational Companion — Kronecker Products, SVD, Singlet Correlations, and CHSH

This notebook verifies every claim in Lecture 6:

1. **Kronecker product** — construct composite states and basis vectors
2. **Coefficient matrix and rank** — product states (rank 1) vs entangled (rank 2)
3. **Schmidt decomposition = SVD** — singular values quantify entanglement
4. **Parameter counting** — most random states are entangled
5. **Singlet correlations** — simultaneous eigenvector of X⊗X, Y⊗Y, Z⊗Z
6. **CHSH inequality** — quantum value 2√2 violates classical bound of 2
7. **Von Neumann measurement model** — collapse from entanglement

**Convention:** ℏ = 1 throughout.

**Reference:** Lecture 6 notes ("Quantum Systems via Linear Algebra")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

# ── Pauli matrices ──
I2 = np.eye(2, dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)

# ── Basis vectors ──
e0 = np.array([1, 0], dtype=complex)
e1 = np.array([0, 1], dtype=complex)

def kron(a, b):
    """Kronecker product of two vectors or matrices."""
    return np.kron(a, b)

def expect(M, v):
    """Expectation value <v|M|v> for a vector v."""
    return np.real(v.conj() @ M @ v)

def coeff_matrix(v4):
    """Reshape a 4-vector into the 2x2 coefficient matrix Ψ."""
    return v4.reshape(2, 2)

print("Setup complete: Pauli matrices, Kronecker product, helpers.")

## 1. Kronecker Product — Building the Composite Space

The combined state space is $\mathbb{C}^2 \otimes \mathbb{C}^2 = \mathbb{C}^4$ with basis
$\{\mathbf{e}_0 \otimes \mathbf{f}_0,\; \mathbf{e}_0 \otimes \mathbf{f}_1,\; \mathbf{e}_1 \otimes \mathbf{f}_0,\; \mathbf{e}_1 \otimes \mathbf{f}_1\}$.

In [ ]:
# ── 1. Kronecker Product ──

print("=" * 65)
print("KRONECKER PRODUCT: composite basis and inner product")
print("=" * 65)

# Composite basis vectors
basis = [kron(e0, e0), kron(e0, e1), kron(e1, e0), kron(e1, e1)]
labels = ['e0⊗f0', 'e0⊗f1', 'e1⊗f0', 'e1⊗f1']

print("\nComposite basis vectors:")
for lbl, b in zip(labels, basis):
    print(f"  {lbl} = {b}")

# Verify orthonormality
print("\nOrthonormality check (inner product matrix):")
G = np.array([[basis[i].conj() @ basis[j] for j in range(4)] for i in range(4)])
print(np.round(G.real, 4))
print(f"Is identity: {np.allclose(G, np.eye(4))}")

# Kronecker product of operators
print("\nZ ⊗ Z (4×4 matrix):")
ZZ = kron(Z, Z)
print(ZZ)

## 2. Coefficient Matrix and Rank — Product vs Entangled

Write the 4 coefficients as a 2×2 matrix $\Psi$. Product state ⟺ rank($\Psi$) = 1. Entangled ⟺ rank($\Psi$) ≥ 2.

In [ ]:
# ── 2. Coefficient Matrix and Rank ──

print("=" * 65)
print("COEFFICIENT MATRIX: rank determines entanglement")
print("=" * 65)

# Product state: e0 ⊗ x+ = e0 ⊗ (e0+e1)/√2
x_plus = (e0 + e1) / np.sqrt(2)
v_product = kron(e0, x_plus)
Psi_prod = coeff_matrix(v_product)
print(f"\nProduct state: e0 ⊗ x+")
print(f"Coefficient matrix Ψ =\n{np.round(Psi_prod, 4)}")
print(f"rank(Ψ) = {np.linalg.matrix_rank(Psi_prod)}")
print(f"det(Ψ) = {np.linalg.det(Psi_prod):.6f}")

# Singlet state
v_singlet = (kron(e0, e1) - kron(e1, e0)) / np.sqrt(2)
Psi_sing = coeff_matrix(v_singlet)
print(f"\nSinglet state: (e0⊗f1 − e1⊗f0)/√2")
print(f"Coefficient matrix Ψ =\n{np.round(Psi_sing, 4)}")
print(f"rank(Ψ) = {np.linalg.matrix_rank(Psi_sing)}")
print(f"det(Ψ) = {np.linalg.det(Psi_sing):.6f}  ≠ 0 → rank 2 ✓")

# Verify Ψ†Ψ = ½I for singlet
PdP = Psi_sing.conj().T @ Psi_sing
print(f"\nΨ†Ψ (singlet) =\n{np.round(PdP, 4)}")
print(f"Is ½I: {np.allclose(PdP, 0.5 * np.eye(2))}")

## 3. Schmidt Decomposition = SVD

The Schmidt decomposition is literally the SVD of the coefficient matrix $\Psi = U \Sigma V^\dagger$. The singular values quantify entanglement:
- **Product**: one nonzero σ
- **Maximally entangled**: equal σ's ($\sigma_1 = \sigma_2 = 1/\sqrt{2}$)
- **Partially entangled**: unequal nonzero σ's

In [ ]:
# ── 3. Schmidt Decomposition = SVD ──

print("=" * 65)
print("SCHMIDT DECOMPOSITION = SVD of coefficient matrix")
print("=" * 65)

test_states = [
    ("Product: e0 ⊗ x+", kron(e0, x_plus)),
    ("Singlet", v_singlet),
    ("Partially entangled", (np.sqrt(0.8)*kron(e0,e0) + np.sqrt(0.2)*kron(e1,e1))),
    ("Bell |Φ+⟩", (kron(e0,e0) + kron(e1,e1)) / np.sqrt(2)),
]

for name, v in test_states:
    Psi = coeff_matrix(v)
    U, sigmas, Vh = np.linalg.svd(Psi)
    print(f"\n  {name}:")
    print(f"    Singular values: {np.round(sigmas, 6)}")
    print(f"    # nonzero σ's: {np.sum(sigmas > 1e-10)}")
    if np.sum(sigmas > 1e-10) == 1:
        print(f"    → Product state")
    elif np.allclose(sigmas[0], sigmas[1]):
        print(f"    → Maximally entangled (equal σ's)")
    else:
        print(f"    → Partially entangled")

## 4. Parameter Counting — Most States Are Entangled

A general 2-qubit state has **6 real parameters** (8 real from 4 complex, minus normalization and phase). A product state has only **4** (2 per subsystem). Since 6 > 4, product states form a lower-dimensional subset: **most states are entangled**.

We verify by generating random states and checking what fraction are product states.

In [ ]:
# ── 4. Parameter Counting: Most States Are Entangled ──

print("=" * 65)
print("PARAMETER COUNTING: most random states are entangled")
print("=" * 65)

rng = np.random.default_rng(42)
n_samples = 10000
sigmas_all = np.zeros((n_samples, 2))

for i in range(n_samples):
    # Random state: sample from Haar measure (random complex Gaussian, normalize)
    v = rng.standard_normal(4) + 1j * rng.standard_normal(4)
    v /= np.linalg.norm(v)
    Psi = coeff_matrix(v)
    _, s, _ = np.linalg.svd(Psi)
    sigmas_all[i] = s

# A state is product if the smaller singular value is ~0
entanglement_ratio = sigmas_all[:, 1] / sigmas_all[:, 0]
product_count = np.sum(sigmas_all[:, 1] < 1e-6)
max_ent_count = np.sum(np.abs(sigmas_all[:, 0] - sigmas_all[:, 1]) < 0.01)

print(f"\n{n_samples} random Haar states:")
print(f"  Product states (σ₂ < 1e-6): {product_count} ({100*product_count/n_samples:.2f}%)")
print(f"  Near-maximally entangled (|σ₁−σ₂| < 0.01): {max_ent_count} ({100*max_ent_count/n_samples:.2f}%)")
print(f"  Entangled: {n_samples - product_count} ({100*(n_samples-product_count)/n_samples:.2f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].hist(sigmas_all[:, 1], bins=80, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--', linewidth=2, label='Product (σ₂=0)')
axes[0].axvline(1/np.sqrt(2), color='green', linestyle='--', linewidth=2, label=f'Max entangled (σ₂=1/√2≈{1/np.sqrt(2):.3f})')
axes[0].set_xlabel('Smaller singular value σ₂'); axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of σ₂ for 10,000 Haar-random states')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

axes[1].scatter(sigmas_all[:, 0], sigmas_all[:, 1], alpha=0.1, s=3, c='steelblue')
axes[1].plot([0, 1], [0, 1], 'g--', label='σ₁=σ₂ (max entangled)')
axes[1].axhline(0, color='r', linestyle='--', label='σ₂=0 (product)')
axes[1].set_xlabel('σ₁'); axes[1].set_ylabel('σ₂')
axes[1].set_title('Singular value pairs (σ₁, σ₂)')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
axes[1].set_xlim(0, 1.05); axes[1].set_ylim(-0.05, 0.85)

plt.suptitle('Most states are entangled: product states have measure zero', fontsize=13)
plt.tight_layout(); plt.show()

## 5. Singlet Correlations — Simultaneous Eigenvector

The singlet is a simultaneous eigenvector of $X \otimes X$, $Y \otimes Y$, and $Z \otimes Z$, all with eigenvalue $-1$. This is its rotational invariance.

We also verify: individual expectations vanish, matching expectations give $-1$, cross expectations give $0$.

In [ ]:
# ── 5. Singlet Correlations ──

print("=" * 65)
print("SINGLET CORRELATIONS: simultaneous eigenvector of M⊗M")
print("=" * 65)

# 4×4 composite operators
XX = kron(X, X)
YY = kron(Y, Y)
ZZ = kron(Z, Z)
XI = kron(X, I2)
YI = kron(Y, I2)
ZI = kron(Z, I2)
IX = kron(I2, X)
IY = kron(I2, Y)
IZ = kron(I2, Z)
XZ = kron(X, Z)
ZX = kron(Z, X)

print("\nIndividual expectations (should all be 0):")
for name, op in [('⟨Z⊗I⟩', ZI), ('⟨I⊗Z⟩', IZ), ('⟨X⊗I⟩', XI), ('⟨I⊗X⟩', IX), ('⟨Y⊗I⟩', YI), ('⟨I⊗Y⟩', IY)]:
    print(f"  {name} = {expect(op, v_singlet):.6f}")

print("\nMatching observable correlations (should all be -1):")
for name, op in [('⟨Z⊗Z⟩', ZZ), ('⟨X⊗X⟩', XX), ('⟨Y⊗Y⟩', YY)]:
    val = expect(op, v_singlet)
    print(f"  {name} = {val:.6f}")

print("\nCross-observable correlations (should all be 0):")
for name, op in [('⟨X⊗Z⟩', XZ), ('⟨Z⊗X⟩', ZX), ('⟨X⊗Y⟩', kron(X,Y)), ('⟨Y⊗Z⟩', kron(Y,Z))]:
    print(f"  {name} = {expect(op, v_singlet):.6f}")

# Verify singlet is EIGENVECTOR of each M⊗M with eigenvalue -1
print("\nEigenvector check: (M⊗M)|singlet⟩ = -|singlet⟩")
for name, op in [('X⊗X', XX), ('Y⊗Y', YY), ('Z⊗Z', ZZ)]:
    result = op @ v_singlet
    ratio = result / v_singlet
    eigenvalue = ratio[np.argmax(np.abs(v_singlet))]
    print(f"  ({name})|singlet⟩ = {np.real(eigenvalue):.1f} × |singlet⟩  ✓")

## 6. CHSH Inequality — Quantum Beats Classical

The CHSH quantity is $S = \langle A_0 B_0 \rangle + \langle A_0 B_1 \rangle + \langle A_1 B_0 \rangle - \langle A_1 B_1 \rangle$.

- **Classical bound**: $|S| \leq 2$ (local hidden variable theories)
- **Quantum (singlet)**: $S = 2\sqrt{2} \approx 2.828$ (Tsirelson bound)

We compute this with $A_0 = Z$, $A_1 = X$ for Alice, and $B_0 = -(Z+X)/\sqrt{2}$, $B_1 = (Z-X)/\sqrt{2}$ for Bob.

In [ ]:
# ── 6. CHSH Inequality ──

print("=" * 65)
print("CHSH INEQUALITY: quantum value 2√2 > classical bound 2")
print("=" * 65)

# Alice's observables
A0 = Z
A1 = X

# Bob's observables (rotated by 45°)
B0 = -(Z + X) / np.sqrt(2)
B1 = (Z - X) / np.sqrt(2)

# Verify these are valid observables (Hermitian, eigenvalues ±1)
for name, op in [('A0', A0), ('A1', A1), ('B0', B0), ('B1', B1)]:
    evals = np.linalg.eigvalsh(op)
    print(f"  {name} eigenvalues: {np.round(evals, 4)} (should be ±1)")

# Composite operators
A0B0 = kron(A0, B0)
A0B1 = kron(A0, B1)
A1B0 = kron(A1, B0)
A1B1 = kron(A1, B1)

# Individual correlations
c00 = expect(A0B0, v_singlet)
c01 = expect(A0B1, v_singlet)
c10 = expect(A1B0, v_singlet)
c11 = expect(A1B1, v_singlet)

print(f"\n  ⟨A₀B₀⟩ = {c00:.6f}")
print(f"  ⟨A₀B₁⟩ = {c01:.6f}")
print(f"  ⟨A₁B₀⟩ = {c10:.6f}")
print(f"  ⟨A₁B₁⟩ = {c11:.6f}")

S = c00 + c01 + c10 - c11
print(f"\n  CHSH quantity S = {S:.6f}")
print(f"  2√2 = {2*np.sqrt(2):.6f}")
print(f"  S = 2√2? {np.isclose(S, 2*np.sqrt(2))}")
print(f"\n  Classical bound: |S| ≤ 2")
print(f"  Quantum value:   |S| = {abs(S):.4f} > 2  →  Bell inequality VIOLATED ✓")
print(f"\n  This proves no local hidden variable theory can reproduce quantum predictions.")

# Sweep over angle to show Tsirelson bound is the maximum
thetas = np.linspace(0, np.pi, 200)
S_vals = []
for th in thetas:
    B0_th = -(np.cos(th) * Z + np.sin(th) * X)
    B1_th = (np.cos(th) * Z - np.sin(th) * X)
    s = (expect(kron(A0, B0_th), v_singlet) +
         expect(kron(A0, B1_th), v_singlet) +
         expect(kron(A1, B0_th), v_singlet) -
         expect(kron(A1, B1_th), v_singlet))
    S_vals.append(s)

plt.figure(figsize=(10, 5))
plt.plot(np.degrees(thetas), S_vals, 'steelblue', linewidth=2, label='CHSH value S(θ)')
plt.axhline(2, color='red', linestyle='--', linewidth=2, label='Classical bound = 2')
plt.axhline(2*np.sqrt(2), color='green', linestyle=':', linewidth=2, label=f'Tsirelson bound = 2√2 ≈ {2*np.sqrt(2):.3f}')
plt.xlabel("Bob's rotation angle θ (degrees)"); plt.ylabel('S')
plt.title('CHSH quantity vs Bob\'s measurement angle\nSinglet state, Alice measures Z and X')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Von Neumann Measurement Model — Collapse from Entanglement

A system in state $\alpha|0\rangle + \beta|1\rangle$ interacts with an apparatus initially in $|f_0\rangle$. The unitary copies the system state into the apparatus pointer:

$$U(\alpha|e_0\rangle + \beta|e_1\rangle) \otimes |f_0\rangle = \alpha|e_0\rangle\otimes|f_0\rangle + \beta|e_1\rangle\otimes|f_1\rangle$$

The result is entangled — "collapse" is just Bayesian conditioning on the apparatus reading.

In [ ]:
# ── 7. Von Neumann Measurement Model ──

print("=" * 65)
print("VON NEUMANN MEASUREMENT: collapse from entanglement")
print("=" * 65)

# System state: α|e0⟩ + β|e1⟩
alpha = np.cos(np.pi/5)
beta = np.exp(1j * np.pi/3) * np.sin(np.pi/5)
psi_sys = alpha * e0 + beta * e1
print(f"\nSystem state: α={alpha:.4f}, β={beta:.4f}")
print(f"|α|² = {abs(alpha)**2:.4f}, |β|² = {abs(beta)**2:.4f}")

# Apparatus initial state: |f0⟩
f0, f1 = e0, e1  # apparatus uses same basis

# Pre-measurement: |ψ⟩ ⊗ |f0⟩
v_pre = kron(psi_sys, f0)
Psi_pre = coeff_matrix(v_pre)
print(f"\nPre-measurement coefficient matrix:")
print(f"{np.round(Psi_pre, 4)}")
print(f"rank = {np.linalg.matrix_rank(Psi_pre)} (product state ✓)")

# Build the CNOT-like unitary: U|e0⟩|f0⟩ = |e0⟩|f0⟩, U|e1⟩|f0⟩ = |e1⟩|f1⟩
# This is the CNOT gate
U_meas = np.zeros((4, 4), dtype=complex)
U_meas[0, 0] = 1  # |00⟩ → |00⟩
U_meas[1, 1] = 1  # |01⟩ → |01⟩
U_meas[3, 2] = 1  # |10⟩ → |11⟩
U_meas[2, 3] = 1  # |11⟩ → |10⟩
print(f"\nMeasurement unitary U (CNOT):")
print(U_meas.real.astype(int))
print(f"Unitary check: U†U = I? {np.allclose(U_meas.conj().T @ U_meas, np.eye(4))}")

# Post-measurement state
v_post = U_meas @ v_pre
Psi_post = coeff_matrix(v_post)
print(f"\nPost-measurement coefficient matrix:")
print(f"{np.round(Psi_post, 4)}")
print(f"rank = {np.linalg.matrix_rank(Psi_post)}")
_, s_post, _ = np.linalg.svd(Psi_post)
print(f"Singular values: {np.round(s_post, 4)}")
if np.linalg.matrix_rank(Psi_post) > 1:
    print("→ ENTANGLED! System and apparatus are correlated.")

# Verify: coefficients match α, β
print(f"\nPost-measurement state:")
print(f"  α(e0⊗f0) + β(e1⊗f1) = {np.round(alpha, 4)}|00⟩ + {np.round(beta, 4)}|11⟩")
print(f"  Actual: {np.round(v_post, 4)}")
expected_post = alpha * kron(e0, f0) + beta * kron(e1, f1)
print(f"  Match: {np.allclose(v_post, expected_post)}")

# Conditional states
print(f"\nConditional states (\"collapse\"):")
print(f"  If apparatus reads f0 (prob |α|²={abs(alpha)**2:.4f}): system → e0")
print(f"  If apparatus reads f1 (prob |β|²={abs(beta)**2:.4f}): system → e1")
print(f"\n→ \"Collapse\" is just Bayesian conditioning on the apparatus reading.")

## 8. Subsystem Expectation Values — Product vs Entangled

In a product state, subsystem observables factorize: $\langle L \otimes M \rangle = \langle L \rangle \langle M \rangle$.
In an entangled state, this fails — the correlation $C(L,M) \neq 0$.

In [ ]:
# ── 8. Product vs Entangled: Correlations ──

print("=" * 65)
print("CORRELATIONS: product state (C=0) vs entangled (C≠0)")
print("=" * 65)

# Product state
v_prod = kron(e0, x_plus)
print("\nProduct state: e0 ⊗ x+")
for name_A, opA, name_B, opB in [
    ('Z', Z, 'X', X), ('Z', Z, 'Z', Z), ('X', X, 'X', X)
]:
    joint = expect(kron(opA, opB), v_prod)
    indiv_A = expect(kron(opA, I2), v_prod)
    indiv_B = expect(kron(I2, opB), v_prod)
    C = joint - indiv_A * indiv_B
    print(f"  ⟨{name_A}⊗{name_B}⟩={joint:.4f}, ⟨{name_A}⟩⟨{name_B}⟩={indiv_A*indiv_B:.4f}, C={C:.6f}")

# Singlet
print("\nSinglet state:")
for name_A, opA, name_B, opB in [
    ('Z', Z, 'Z', Z), ('X', X, 'X', X), ('Y', Y, 'Y', Y),
    ('X', X, 'Z', Z), ('Z', Z, 'X', X)
]:
    joint = expect(kron(opA, opB), v_singlet)
    indiv_A = expect(kron(opA, I2), v_singlet)
    indiv_B = expect(kron(I2, opB), v_singlet)
    C = joint - indiv_A * indiv_B
    print(f"  ⟨{name_A}⊗{name_B}⟩={joint:.4f}, ⟨{name_A}⟩⟨{name_B}⟩={indiv_A*indiv_B:.4f}, C={C:.6f}")